In [1]:
from kaggle_handler import handler
import pandas as pd
import numpy as np

In [9]:
Assets = handler("fredericobreno/play-tennis", Add_more=False, Folder_Name="Play_Tennis")

Directory 'Assets' already exists.
Directory 'Play_Tennis' already exists.
Datasets already exist in Assets folder
['HeartDisease', 'Play_Tennis']
 Change Add_more parameter to download more datasets


In [3]:
!ls Assets/Play_Tennis

play_tennis.csv


In [4]:
df = pd.read_csv("Assets/Play_Tennis/play_tennis.csv")
df.sample(5)

,day,outlook,temp,humidity,wind,play
8,D9,Sunny,Cool,Normal,Weak,Yes
11,D12,Overcast,Mild,High,Strong,Yes
12,D13,Overcast,Hot,Normal,Weak,Yes
9,D10,Rain,Mild,Normal,Weak,Yes
7,D8,Sunny,Mild,High,Weak,No


In [5]:
df.shape

(14, 6)

In [6]:
class MyNaivBayes:
    def __init__(self):
        self.Pivotdata: dict = {}
        self.n_classes = None
        pass

    def train(self, X, y):
        No_of_obs = X.shape[0]
        
        self.n_classes = self.N_Classes(y)
        class_group_index = self.Class_group(y, self.n_classes)
        self.Probability(No_of_obs, class_group_index)
        self.feature_probability(X, class_group_index)
        
    def N_Classes(self, data):
        return np.unique(data)

    def Class_group(self, data, n_classes):
        class_group: dict = {}
        for cls in n_classes:
            class_group[cls] = np.where(data==cls)[0]
        return class_group

    def Probability(self, no_of_obs, class_group:dict, X_features=None):
        if X_features == None:
            for cls in class_group.keys():
                self.Pivotdata[f'Prob_{cls}'] = len(class_group[cls])/no_of_obs
        else:
            for cls in class_group.keys():
                self.Pivotdata[f'Prob_{X_features}{cls}'] = len(class_group[cls])/no_of_obs

    def feature_probability(self, X, class_group):
        for key in class_group.keys():
            for coll in range(X.shape[1]):
                data = X[class_group[key],coll]
                No_of_obs = data.shape[0]
                n_classes = self.N_Classes(data)
                class_group_index = self.Class_group(data, n_classes)
                self.Probability(No_of_obs, class_group_index, X_features=key)
        
    def predict(self, X_test):
        pred = []
        for dt in X_test:
            outcoms: dict = {}
            for cls in self.n_classes:
                outcoms[cls] = self.Prob_calc(dt, cls)
            pred.append(self.max_prob(outcoms))
        return pred

    def Prob_calc(self, row, cls):
        calc = self.Pivotdata[f'Prob_{cls}']
        for dt in row:
            try:
                calc *= self.Pivotdata[f'Prob_{cls}{dt}']
            except:
                calc *= 0
        return calc

    def max_prob(self, outcoms):
        value = np.max(list(outcoms.values()))
        for key in outcoms.keys():
            if outcoms[key] == value:
                return key

In [7]:
model = MyNaivBayes()
X = np.array(df.drop(columns=['day','play']))
y = np.array(df[["play"]])
model.train(X, y)

In [8]:
X_test = np.array([["Sunny", "Cool", "Normal", "Weak"],["Overcast", "Mild", "High", "Strong"]])
model.predict(X_test)

['Yes', 'Yes']